In [0]:
-- Paso 1: MERGE incremental para la tabla suscripciones_historico
-- Carga histórica con id_suscripcion y fecha_extraccion como claves
MERGE INTO IDENTIFIER(:catalogo || '.core_negocio_' || :alumno || '.suscripciones_historico') AS target
USING (
    SELECT
        CAST(subscription_id AS BIGINT) AS id_suscripcion,
        subscription_name AS nombre_suscripcion,
        CAST(max_contents_per_month AS INT) AS max_contenidos_mensuales,
        CAST(created_at AS TIMESTAMP) AS fecha_creacion,
        CAST(updated_at AS TIMESTAMP) AS fecha_actualizacion,
        fecha_extraccion
    FROM IDENTIFIER(:catalogo_origen || '.datavision_' || :alumno || '.subscriptions')
    WHERE fecha_extraccion = :fecha_extraccion
) AS source
ON target.id_suscripcion = source.id_suscripcion 
   AND target.fecha_extraccion = source.fecha_extraccion
WHEN MATCHED THEN
    UPDATE SET
        target.nombre_suscripcion = source.nombre_suscripcion,
        target.max_contenidos_mensuales = source.max_contenidos_mensuales,
        target.fecha_creacion = source.fecha_creacion,
        target.fecha_actualizacion = source.fecha_actualizacion
WHEN NOT MATCHED THEN
    INSERT (
        id_suscripcion,
        nombre_suscripcion,
        max_contenidos_mensuales,
        fecha_creacion,
        fecha_actualizacion,
        fecha_extraccion
    )
    VALUES (
        source.id_suscripcion,
        source.nombre_suscripcion,
        source.max_contenidos_mensuales,
        source.fecha_creacion,
        source.fecha_actualizacion,
        source.fecha_extraccion
    );

In [0]:
-- Paso 2: OVERWRITE completo de la tabla suscripciones
-- Solo se ejecuta si el MERGE anterior fue exitoso
INSERT OVERWRITE TABLE IDENTIFIER(:catalogo || '.core_negocio_' || :alumno || '.suscripciones')
SELECT
    CAST(subscription_id AS BIGINT) AS id_suscripcion,
    subscription_name AS nombre_suscripcion,
    CAST(max_contents_per_month AS INT) AS max_contenidos_mensuales,
    CAST(created_at AS TIMESTAMP) AS fecha_creacion,
    CAST(updated_at AS TIMESTAMP) AS fecha_actualizacion
FROM IDENTIFIER(:catalogo_origen || '.datavision_' || :alumno || '.subscriptions')
WHERE fecha_extraccion = :fecha_extraccion;

In [0]:
-- Verificar la cantidad de registros en suscripciones_historico
SELECT
    'suscripciones_historico' as tabla,
    COUNT(*) as total_registros,
    COUNT(DISTINCT id_suscripcion) as suscripciones_unicas,
    COUNT(DISTINCT fecha_extraccion) as fechas_extraccion,
    MAX(fecha_actualizacion) as ultima_actualizacion
FROM IDENTIFIER(:catalogo || '.core_negocio_' || :alumno || '.suscripciones_historico')

UNION ALL

-- Verificar la cantidad de registros en suscripciones
SELECT
    'suscripciones' as tabla,
    COUNT(*) as total_registros,
    COUNT(DISTINCT id_suscripcion) as suscripciones_unicas,
    NULL as fechas_extraccion,
    MAX(fecha_actualizacion) as ultima_actualizacion
FROM IDENTIFIER(:catalogo || '.core_negocio_' || :alumno || '.suscripciones');

In [0]:
-- Mostrar una muestra de los datos de suscripciones_historico
SELECT 'suscripciones_historico' as origen, *
FROM IDENTIFIER(:catalogo || '.core_negocio_' || :alumno || '.suscripciones_historico')
WHERE fecha_extraccion = :fecha_extraccion
ORDER BY fecha_actualizacion DESC
LIMIT 5;

In [0]:
-- Mostrar una muestra de los datos de suscripciones
SELECT 'suscripciones' as origen, *
FROM IDENTIFIER(:catalogo || '.core_negocio_' || :alumno || '.suscripciones')
ORDER BY fecha_actualizacion DESC
LIMIT 5;